# Notebook 3: Evaluate BFCL Results

This notebook evaluates raw model outputs from hardware inference runs.

It:
- loads the fixed BFCL subset
- loads one hardware result file
- parses Qwen `<tool_call>` outputs
- compares predicted tool calls against BFCL gold answers
- computes success rate by category
- saves a scored result file

In [41]:
import json
import re
from pathlib import Path
from collections import Counter

import pandas as pd

In [115]:
DATA_DIR = Path("../data")
RESULTS_RAW_DIR = Path("../results/raw")
RESULTS_PROCESSED_DIR = Path("../results/processed")
RESULTS_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

SUBSET_PATH = DATA_DIR / "bfcl_subset.json"

# Change only RUN_NAME when evaluating a different result file
# RUN_NAME = "Groq_LPU_Llama"
RUN_NAME = "T4"

RAW_RESULTS_PATH = RESULTS_RAW_DIR / f"{RUN_NAME}_results.jsonl"
SCORED_RESULTS_PATH = RESULTS_PROCESSED_DIR / f"{RUN_NAME}_scored.csv"
SUMMARY_PATH = RESULTS_PROCESSED_DIR / f"{RUN_NAME}_category_summary.csv"
FAILURE_TAXONOMY_PATH = RESULTS_PROCESSED_DIR / f"{RUN_NAME}_failure_taxonomy.csv"

print("Raw:", RAW_RESULTS_PATH)
print("Scored:", SCORED_RESULTS_PATH)
print("Summary:", SUMMARY_PATH)
print("Failure taxonomy:", FAILURE_TAXONOMY_PATH)

Raw: ../results/raw/T4_results.jsonl
Scored: ../results/processed/T4_scored.csv
Summary: ../results/processed/T4_category_summary.csv
Failure taxonomy: ../results/processed/T4_failure_taxonomy.csv


In [116]:
with open(SUBSET_PATH, "r", encoding="utf-8") as f:
    tasks = json.load(f)

task_lookup = {t["task_id"]: t for t in tasks}

print("Loaded tasks:", len(tasks))
print("Categories:", Counter(t["category"] for t in tasks))

Loaded tasks: 100
Categories: Counter({'simple': 25, 'multiple': 25, 'parallel': 25, 'parallel_multiple': 25})


In [117]:
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

raw_results = load_jsonl(RAW_RESULTS_PATH)

print("Loaded raw result rows:", len(raw_results))
pd.DataFrame(raw_results).head()

Loaded raw result rows: 100


,task_id,category,hardware,model,backend,run_index,raw_output,latency_s,input_tokens,output_tokens,tokens_per_second,peak_memory_mb,error,timestamp
0,simple_361,simple,T4,Qwen/Qwen2.5-7B-Instruct,transformers,0,"<tool_call>\n{""name"": ""restaurant_finder"", ""ar...",34.944954,257,36,1.030192,13219.127441,None,2026-04-28T22:39:19.708422
1,simple_113,simple,T4,Qwen/Qwen2.5-7B-Instruct,transformers,0,"<tool_call>\n{""name"": ""probability.dice_roll"",...",27.599000,269,40,1.449328,13220.002441,None,2026-04-28T22:39:54.665027
2,simple_135,simple,T4,Qwen/Qwen2.5-7B-Instruct,transformers,0,"<tool_call>\n{""name"": ""calculate_return_on_inv...",27.128561,267,40,1.474461,13219.893066,None,2026-04-28T22:40:22.269049
3,simple_1,simple,T4,Qwen/Qwen2.5-7B-Instruct,transformers,0,"<tool_call>\n{""name"": ""math.factorial"", ""argum...",14.107095,179,21,1.488613,13214.038574,None,2026-04-28T22:40:49.400034
4,simple_123,simple,T4,Qwen/Qwen2.5-7B-Instruct,transformers,0,"<tool_call>\n{""name"": ""hypothesis_testing.two_...",61.864661,349,84,1.357803,13226.786621,None,2026-04-28T22:41:03.509655


## Tool Call Parsing

Qwen often returns tool calls in this format:

`<tool_call>{"name": "...", "arguments": {...}}</tool_call>`

The parser extracts one or more JSON tool-call objects from the model output.

In [118]:
def parse_tool_calls(raw_output):
    """
    Parse Qwen-style tool calls from raw model output.

    Expected format:
    <tool_call>
    {"name": "function_name", "arguments": {...}}
    </tool_call>

    Returns:
    - list of {"name": str, "arguments": dict}
    - None if parsing fails
    """
    if raw_output is None:
        return None

    text = raw_output.strip()

    # Extract content inside <tool_call>...</tool_call>
    matches = re.findall(r"<tool_call>\s*(.*?)\s*</tool_call>", text, flags=re.DOTALL)

    parsed = []

    # If no tags found, try to parse entire output as JSON fallback
    if not matches:
        matches = [text]

    for m in matches:
        m = m.strip()

        # Remove accidental markdown fences
        m = re.sub(r"^```json\s*", "", m)
        m = re.sub(r"^```\s*", "", m)
        m = re.sub(r"\s*```$", "", m)

        try:
            obj = json.loads(m)

            if isinstance(obj, dict):
                # Qwen-style: {"name": ..., "arguments": ...}
                if "name" in obj and "arguments" in obj:
                    parsed.append({
                        "name": obj["name"],
                        "arguments": obj.get("arguments", {})
                    })

                # Sometimes model may emit {"function": ..., "arguments": ...}
                elif "function" in obj and "arguments" in obj:
                    parsed.append({
                        "name": obj["function"],
                        "arguments": obj.get("arguments", {})
                    })

            elif isinstance(obj, list):
                for item in obj:
                    if isinstance(item, dict) and "name" in item:
                        parsed.append({
                            "name": item["name"],
                            "arguments": item.get("arguments", {})
                        })

        except Exception:
            continue

    return parsed if parsed else None

In [119]:
for row in raw_results[:5]:
    print("TASK:", row["task_id"])
    print("RAW:", row["raw_output"][:300])
    print("PARSED:", parse_tool_calls(row["raw_output"]))
    print("-" * 80)

TASK: simple_361
RAW: <tool_call>
{"name": "restaurant_finder", "arguments": {"city": "New York", "cuisine": "Italian", "diet": "Gluten-free"}}
</tool_call>
PARSED: [{'name': 'restaurant_finder', 'arguments': {'city': 'New York', 'cuisine': 'Italian', 'diet': 'Gluten-free'}}]
--------------------------------------------------------------------------------
TASK: simple_113
RAW: <tool_call>
{"name": "probability.dice_roll", "arguments": {"desired_number": 6, "number_of_rolls": 2, "die_sides": 6}}
</tool_call>
PARSED: [{'name': 'probability.dice_roll', 'arguments': {'desired_number': 6, 'number_of_rolls': 2, 'die_sides': 6}}]
--------------------------------------------------------------------------------
TASK: simple_135
RAW: <tool_call>
{"name": "calculate_return_on_investment", "arguments": {"purchase_price": 20, "sale_price": 25, "dividend": 2}}
</tool_call>
PARSED: [{'name': 'calculate_return_on_investment', 'arguments': {'purchase_price': 20, 'sale_price': 25, 'dividend': 2}}]
-----

In [120]:
def normalize_gold_calls(gold_answer):
    """
    Convert BFCL gold_answer into list of:
    {"name": function_name, "arguments": {...}}

    Handles gold format:
    [
      {"function_name": {"arg": value}},
      {"another_function": {"arg": value}}
    ]
    """
    if gold_answer is None:
        return None

    normalized = []

    if not isinstance(gold_answer, list):
        return None

    for item in gold_answer:
        if not isinstance(item, dict):
            continue

        for func_name, args in item.items():
            normalized.append({
                "name": func_name,
                "arguments": args if isinstance(args, dict) else {}
            })

    return normalized if normalized else []

## Optional Argument Handling

BFCL function schemas include a `required` list. If an argument is not required by the schema, the scorer should not automatically penalize the model for omitting it.

In [121]:
def is_optional_arg(task, func_name, arg_name):
    """
    Returns True if arg_name is optional according to the function schema.
    Returns False if the function or argument cannot be found.
    """
    tools = task.get("functions_or_tools", [])

    for tool in tools:
        if tool.get("name") == func_name:
            parameters = tool.get("parameters", {})
            required_args = parameters.get("required", [])

            # If arg is not listed as required, it is optional
            return arg_name not in required_args

    return False

In [122]:
def value_matches(pred_value, gold_value):
    """
    Compare predicted argument value against gold value.

    BFCL gold often stores acceptable alternatives as lists.
    Example:
    gold: ["New York City", "NYC"]
    pred: "NYC"
    should match.
    """
    if isinstance(gold_value, list):
        # If gold is list of acceptable scalar values
        if pred_value in gold_value:
            return True

        # If pred is also list, all predicted values should be acceptable
        if isinstance(pred_value, list):
            return all(v in gold_value for v in pred_value)

        # Loose string comparison
        return str(pred_value).strip().lower() in [str(v).strip().lower() for v in gold_value]

    if isinstance(pred_value, list):
        return gold_value in pred_value

    return str(pred_value).strip().lower() == str(gold_value).strip().lower()

In [123]:
def score_calls(pred_calls, gold_calls, task=None):
    """
    Scoring:
    - same number of calls
    - same function names
    - required gold arguments must appear and match
    - optional gold arguments may be omitted
    - allows gold lists as acceptable alternatives

    This is still a simplified scorer, but it is more faithful to BFCL schemas.
    """
    if pred_calls is None:
        return False, "parse_failure: Could not parse predicted tool calls."

    if gold_calls is None:
        return False, "missing_gold: Missing gold calls."

    if len(pred_calls) != len(gold_calls):
        return False, f"wrong_call_count: predicted {len(pred_calls)}, gold {len(gold_calls)}."

    used_pred_indices = set()

    for gold in gold_calls:
        gold_name = gold["name"]
        gold_args = gold.get("arguments", {})

        match_idx = None

        for i, pred in enumerate(pred_calls):
            if i in used_pred_indices:
                continue

            if pred.get("name") == gold_name:
                match_idx = i
                break

        if match_idx is None:
            return False, f"wrong_or_missing_function: Missing expected function '{gold_name}'."

        pred_args = pred_calls[match_idx].get("arguments", {})

        for arg_name, gold_value in gold_args.items():
            if arg_name not in pred_args:
                if task is not None and is_optional_arg(task, gold_name, arg_name):
                    continue

                return False, f"missing_required_argument: Missing argument '{arg_name}' for function '{gold_name}'."

            pred_value = pred_args[arg_name]

            if not value_matches(pred_value, gold_value):
                return False, f"argument_mismatch: Argument mismatch for '{arg_name}': predicted {pred_value}, gold {gold_value}."

        used_pred_indices.add(match_idx)

    return True, "correct: Predicted calls match gold function names and required arguments."

## Score All Results

In [124]:
scored_rows = []

for row in raw_results:
    task_id = row["task_id"]
    task = task_lookup.get(task_id)

    if task is None:
        scored_rows.append({
            **row,
            "correct": False,
            "score_reason": "Task ID not found in BFCL subset.",
            "parsed_calls": None,
            "gold_calls": None
        })
        continue

    pred_calls = parse_tool_calls(row.get("raw_output"))
    gold_calls = normalize_gold_calls(task.get("gold_answer"))

    correct, reason = score_calls(pred_calls, gold_calls, task=task)
    
    scored_rows.append({
        **row,
        "correct": correct,
        "score_reason": reason,
        "parsed_calls": json.dumps(pred_calls, ensure_ascii=False),
        "gold_calls": json.dumps(gold_calls, ensure_ascii=False)
    })

df_scored = pd.DataFrame(scored_rows)

print("Scored rows:", len(df_scored))
df_scored.head()

Scored rows: 100


,task_id,category,hardware,model,backend,run_index,raw_output,latency_s,input_tokens,output_tokens,tokens_per_second,peak_memory_mb,error,timestamp,correct,score_reason,parsed_calls,gold_calls
0,simple_361,simple,T4,Qwen/Qwen2.5-7B-Instruct,transformers,0,"<tool_call>\n{""name"": ""restaurant_finder"", ""ar...",34.944954,257,36,1.030192,13219.127441,None,2026-04-28T22:39:19.708422,True,correct: Predicted calls match gold function n...,"[{""name"": ""restaurant_finder"", ""arguments"": {""...","[{""name"": ""restaurant_finder"", ""arguments"": {""..."
1,simple_113,simple,T4,Qwen/Qwen2.5-7B-Instruct,transformers,0,"<tool_call>\n{""name"": ""probability.dice_roll"",...",27.599000,269,40,1.449328,13220.002441,None,2026-04-28T22:39:54.665027,True,correct: Predicted calls match gold function n...,"[{""name"": ""probability.dice_roll"", ""arguments""...","[{""name"": ""probability.dice_roll"", ""arguments""..."
2,simple_135,simple,T4,Qwen/Qwen2.5-7B-Instruct,transformers,0,"<tool_call>\n{""name"": ""calculate_return_on_inv...",27.128561,267,40,1.474461,13219.893066,None,2026-04-28T22:40:22.269049,True,correct: Predicted calls match gold function n...,"[{""name"": ""calculate_return_on_investment"", ""a...","[{""name"": ""calculate_return_on_investment"", ""a..."
3,simple_1,simple,T4,Qwen/Qwen2.5-7B-Instruct,transformers,0,"<tool_call>\n{""name"": ""math.factorial"", ""argum...",14.107095,179,21,1.488613,13214.038574,None,2026-04-28T22:40:49.400034,True,correct: Predicted calls match gold function n...,"[{""name"": ""math.factorial"", ""arguments"": {""num...","[{""name"": ""math.factorial"", ""arguments"": {""num..."
4,simple_123,simple,T4,Qwen/Qwen2.5-7B-Instruct,transformers,0,"<tool_call>\n{""name"": ""hypothesis_testing.two_...",61.864661,349,84,1.357803,13226.786621,None,2026-04-28T22:41:03.509655,True,correct: Predicted calls match gold function n...,"[{""name"": ""hypothesis_testing.two_sample_t_tes...","[{""name"": ""hypothesis_testing.two_sample_t_tes..."


In [125]:
overall_success = df_scored["correct"].mean()

print("Overall success rate:", overall_success)
print("Correct:", df_scored["correct"].sum())
print("Total:", len(df_scored))

Overall success rate: 0.91
Correct: 91
Total: 100


In [126]:
category_summary = df_scored.groupby("category").agg(
    n=("correct", "count"),
    success_rate=("correct", "mean"),
    median_latency_s=("latency_s", "median"),
    mean_latency_s=("latency_s", "mean"),
    mean_tokens_per_second=("tokens_per_second", "mean"),
    mean_output_tokens=("output_tokens", "mean")
).reset_index()

category_summary

,category,n,success_rate,median_latency_s,mean_latency_s,mean_tokens_per_second,mean_output_tokens
0,multiple,25,1.00,24.106324,25.818953,1.470994,38.00
1,parallel,25,0.80,71.560449,76.934525,1.481991,113.96
2,parallel_multiple,25,0.88,72.341025,77.880363,1.479388,115.12
3,simple,25,0.96,27.528042,27.982149,1.458864,40.52


## Wilson Confidence Intervals

Because each category has only 25 tasks, we compute 95% Wilson confidence intervals for success rates.

In [127]:
def wilson_ci(successes, n, z=1.96):
    if n == 0:
        return None, None

    p = successes / n
    denominator = 1 + (z**2 / n)
    center = (p + (z**2 / (2 * n))) / denominator
    margin = (
        z
        * ((p * (1 - p) / n + z**2 / (4 * n**2)) ** 0.5)
        / denominator
    )

    return center - margin, center + margin

In [128]:
category_summary_ci = []

for category, group in df_scored.groupby("category"):
    n = len(group)
    successes = int(group["correct"].sum())
    success_rate = successes / n

    ci_low, ci_high = wilson_ci(successes, n)

    category_summary_ci.append({
        "category": category,
        "n": n,
        "successes": successes,
        "success_rate": success_rate,
        "ci_low": ci_low,
        "ci_high": ci_high,
        "median_latency_s": group["latency_s"].median(),
        "mean_latency_s": group["latency_s"].mean(),
        "mean_tokens_per_second": group["tokens_per_second"].mean(),
        "mean_output_tokens": group["output_tokens"].mean()
    })

df_category_ci = pd.DataFrame(category_summary_ci)
df_category_ci

,category,n,successes,success_rate,ci_low,ci_high,median_latency_s,mean_latency_s,mean_tokens_per_second,mean_output_tokens
0,multiple,25,25,1.00,0.866804,1.000000,24.106324,25.818953,1.470994,38.00
1,parallel,25,20,0.80,0.608687,0.911395,71.560449,76.934525,1.481991,113.96
2,parallel_multiple,25,22,0.88,0.700438,0.958333,72.341025,77.880363,1.479388,115.12
3,simple,25,24,0.96,0.804555,0.992904,27.528042,27.982149,1.458864,40.52


In [129]:
overall_successes = int(df_scored["correct"].sum())
overall_n = len(df_scored)
overall_ci_low, overall_ci_high = wilson_ci(overall_successes, overall_n)

print(f"Overall success rate: {overall_successes}/{overall_n} = {overall_successes/overall_n:.3f}")
print(f"95% Wilson CI: [{overall_ci_low:.3f}, {overall_ci_high:.3f}]")

Overall success rate: 91/100 = 0.910
95% Wilson CI: [0.838, 0.952]


In [130]:
failures = df_scored[df_scored["correct"] == False]

print("Failures:", len(failures))

failures[[
    "task_id",
    "category",
    "raw_output",
    "parsed_calls",
    "gold_calls",
    "score_reason"
]].head(10)

Failures: 9


,task_id,category,raw_output,parsed_calls,gold_calls,score_reason
16,simple_260,simple,"<tool_call>\n{""name"": ""paint_requirement.calcu...","[{""name"": ""paint_requirement.calculate"", ""argu...","[{""name"": ""paint_requirement.calculate"", ""argu...",argument_mismatch: Argument mismatch for 'area...
54,parallel_190,parallel,"<tool_call>\n{""name"": ""calculate_shortest_dist...","[{""name"": ""calculate_shortest_distance"", ""argu...","[{""name"": ""calculate_shortest_distance"", ""argu...",argument_mismatch: Argument mismatch for 'star...
55,parallel_183,parallel,"<tool_call>\n{""name"": ""math.hypot"", ""arguments...","[{""name"": ""math.hypot"", ""arguments"": {""x"": 5, ...","[{""name"": ""math.hypot"", ""arguments"": {""x"": [5]...","wrong_call_count: predicted 1, gold 3."
60,parallel_71,parallel,"<tool_call>\n{""name"": ""calculate_derivative"", ...","[{""name"": ""calculate_derivative"", ""arguments"":...","[{""name"": ""calculate_derivative"", ""arguments"":...",argument_mismatch: Argument mismatch for 'func...
63,parallel_185,parallel,"<tool_call>\n{""name"": ""estimate_population"", ""...","[{""name"": ""estimate_population"", ""arguments"": ...","[{""name"": ""estimate_population"", ""arguments"": ...",argument_mismatch: Argument mismatch for 'spec...
65,parallel_105,parallel,"<tool_call>\n{""name"": ""calc_heat_capacity"", ""a...","[{""name"": ""calc_heat_capacity"", ""arguments"": {...","[{""name"": ""calc_heat_capacity"", ""arguments"": {...",argument_mismatch: Argument mismatch for 'temp...
78,parallel_multiple_64,parallel_multiple,"<tool_call>\n{""name"": ""ecological_impact.analy...","[{""name"": ""ecological_impact.analyze"", ""argume...","[{""name"": ""wildlife_population.assess_growth"",...",argument_mismatch: Argument mismatch for 'ecos...
88,parallel_multiple_106,parallel_multiple,"<tool_call>\n{""name"": ""calculate_distance"", ""a...","[{""name"": ""calculate_distance"", ""arguments"": {...","[{""name"": ""traffic_estimate"", ""arguments"": {""s...",argument_mismatch: Argument mismatch for 'time...
94,parallel_multiple_80,parallel_multiple,"<tool_call>\n{""name"": ""functions.intersect"", ""...","[{""name"": ""functions.intersect"", ""arguments"": ...","[{""name"": ""functions.intersect"", ""arguments"": ...",argument_mismatch: Argument mismatch for 'func...


## Failure Mode Taxonomy

We group failed examples by the prefix of the score reason.

In [131]:
def failure_type(reason):
    if not isinstance(reason, str):
        return "unknown"

    if ":" in reason:
        return reason.split(":")[0].strip()

    return "other"

In [132]:
df_scored["failure_type"] = df_scored["score_reason"].apply(
    lambda r: "correct" if str(r).startswith("correct:") else failure_type(r)
)

failure_taxonomy = (
    df_scored[df_scored["correct"] == False]
    .groupby("failure_type")
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

failure_taxonomy

,failure_type,count
0,argument_mismatch,8
1,wrong_call_count,1


In [133]:
failure_by_category = (
    df_scored[df_scored["correct"] == False]
    .groupby(["category", "failure_type"])
    .size()
    .reset_index(name="count")
    .sort_values(["category", "count"], ascending=[True, False])
)

failure_by_category

,category,failure_type,count
0,parallel,argument_mismatch,4
1,parallel,wrong_call_count,1
2,parallel_multiple,argument_mismatch,3
3,simple,argument_mismatch,1


In [134]:
successes = df_scored[df_scored["correct"] == True]

successes[[
    "task_id",
    "category",
    "raw_output",
    "parsed_calls",
    "gold_calls"
]].head(10)

,task_id,category,raw_output,parsed_calls,gold_calls
0,simple_361,simple,"<tool_call>\n{""name"": ""restaurant_finder"", ""ar...","[{""name"": ""restaurant_finder"", ""arguments"": {""...","[{""name"": ""restaurant_finder"", ""arguments"": {""..."
1,simple_113,simple,"<tool_call>\n{""name"": ""probability.dice_roll"",...","[{""name"": ""probability.dice_roll"", ""arguments""...","[{""name"": ""probability.dice_roll"", ""arguments""..."
2,simple_135,simple,"<tool_call>\n{""name"": ""calculate_return_on_inv...","[{""name"": ""calculate_return_on_investment"", ""a...","[{""name"": ""calculate_return_on_investment"", ""a..."
3,simple_1,simple,"<tool_call>\n{""name"": ""math.factorial"", ""argum...","[{""name"": ""math.factorial"", ""arguments"": {""num...","[{""name"": ""math.factorial"", ""arguments"": {""num..."
4,simple_123,simple,"<tool_call>\n{""name"": ""hypothesis_testing.two_...","[{""name"": ""hypothesis_testing.two_sample_t_tes...","[{""name"": ""hypothesis_testing.two_sample_t_tes..."
5,simple_36,simple,"<tool_call>\n{""name"": ""get_shortest_driving_di...","[{""name"": ""get_shortest_driving_distance"", ""ar...","[{""name"": ""get_shortest_driving_distance"", ""ar..."
6,simple_353,simple,"<tool_call>\n{""name"": ""find_recipes"", ""argumen...","[{""name"": ""find_recipes"", ""arguments"": {""diet""...","[{""name"": ""find_recipes"", ""arguments"": {""diet""..."
7,simple_137,simple,"<tool_call>\n{""name"": ""calculate_stock_return""...","[{""name"": ""calculate_stock_return"", ""arguments...","[{""name"": ""calculate_stock_return"", ""arguments..."
8,simple_200,simple,"<tool_call>\n{""name"": ""calculate_emissions"", ""...","[{""name"": ""calculate_emissions"", ""arguments"": ...","[{""name"": ""calculate_emissions"", ""arguments"": ..."
9,simple_179,simple,"<tool_call>\n{""name"": ""find_latest_court_case""...","[{""name"": ""find_latest_court_case"", ""arguments...","[{""name"": ""find_latest_court_case"", ""arguments..."


## Save Scored Results

In [135]:
df_scored.to_csv(SCORED_RESULTS_PATH, index=False)

print(f"Saved scored results to {SCORED_RESULTS_PATH}")

Saved scored results to ../results/processed/T4_scored.csv


In [136]:
df_category_ci.to_csv(SUMMARY_PATH, index=False)

print(f"Saved category summary with CIs to {SUMMARY_PATH}")

Saved category summary with CIs to ../results/processed/T4_category_summary.csv


In [137]:
failure_taxonomy.to_csv(FAILURE_TAXONOMY_PATH, index=False)

print(f"Saved failure taxonomy to {FAILURE_TAXONOMY_PATH}")

Saved failure taxonomy to ../results/processed/T4_failure_taxonomy.csv


In [138]:
FAILURE_BY_CATEGORY_PATH = RESULTS_PROCESSED_DIR / f"{RUN_NAME}_failure_by_category.csv"
failure_by_category.to_csv(FAILURE_BY_CATEGORY_PATH, index=False)

print(f"Saved failure by category to {FAILURE_BY_CATEGORY_PATH}")

Saved failure by category to ../results/processed/T4_failure_by_category.csv
